# DialReader — Stage 2: ANN Digit Classifier (MLP)

Trains a dedicated Fully-Connected Neural Network (Multi-Layer Perceptron) that classifies ONE mechanical dial digit-slot crop (0–9) at a time.

- **Input**: A 64×32 grayscale slot crop, flattened into a **2048-dimensional** vector ($64 \times 32 \times 1$).
- **Architecture**: `Linear(2048 → 256) → BatchNorm → ReLU → Dropout → Linear(256 → 128) → BatchNorm → ReLU → Dropout → Linear(128 → 64) → Linear(64 → 10)`.
- **Execution**: Evaluated 4 times per dial window (Slots 1, 2, 3, 4) to reconstruct the full 4-digit reading (e.g. `"0021"`).
- **Checkpoints**: Automatically saves the best model weights to `models/best_digit_ann.pt` for direct use with `src/infer.py`.

In [ ]:
from pathlib import Path
import sys
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# Resolve dial-reader root directory
PROJECT_ROOT = Path("..").resolve() if (Path("..") / "src").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------------------------
# Hyperparameters & Configuration (Matching train_ann.ipynb style)
# ------------------------------------------------------------------------------
IMG_HEIGHT = 64
IMG_WIDTH = 32
CHANNELS = 1                  # 1 for Grayscale (standardizes black & red drums)
INPUT_DIM = IMG_HEIGHT * IMG_WIDTH * CHANNELS  # 2048
HIDDEN_LAYERS = [256, 128, 64]
NUM_CLASSES = 10              # Digits 0 through 9
DROPOUT = 0.2

BATCH_SIZE = 32
EPOCHS = 35
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

# Set seeds for exact reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device:       {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Input shape:  ({CHANNELS}, {IMG_HEIGHT}, {IMG_WIDTH}) -> Flat Vector: {INPUT_DIM}")
print(f"Architecture: {INPUT_DIM} -> {' -> '.join(str(h) for h in HIDDEN_LAYERS)} -> {NUM_CLASSES}")

--- 
### 2. Dataset Loading (Real Crops + Synthetic Rolling-Drum Expansion)
This dataset loader checks for real crops in `data/digits/`. If real crops are not yet extracted from labels, it automatically generates synthetic rolling-drum digit crops on-the-fly using `SyntheticDigitGenerator` so you can train and evaluate the network immediately.

In [ ]:
from src.synthetic import SyntheticDigitGenerator

# Transforms for ANN digit crops
train_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(degrees=(-8.0, 8.0)),  # Strict: NO horizontal or vertical flips
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

class DialDigitDataset(Dataset):
    """Dataset supporting both saved directory crops and on-the-fly synthetic drum samples."""
    def __init__(self, samples, transform=None):
        self.samples = samples  # List of (PIL.Image or Path, int_label)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item, label = self.samples[idx]
        if isinstance(item, (str, Path)):
            img = Image.open(item).convert("RGB")
        else:
            img = item
        if self.transform:
            img = self.transform(img)
        return img, label

def prepare_datasets():
    digits_dir = PROJECT_ROOT / "data" / "digits"
    train_samples, val_samples = [], []

    # 1. Check for existing real crops
    if (digits_dir / "train").exists():
        for digit_folder in (digits_dir / "train").iterdir():
            if digit_folder.is_dir() and digit_folder.name.isdigit():
                d = int(digit_folder.name)
                for img_path in digit_folder.glob("*.*"):
                    if img_path.suffix.lower() in (".png", ".jpg", ".jpeg"):
                        train_samples.append((img_path, d))

    if (digits_dir / "val").exists():
        for digit_folder in (digits_dir / "val").iterdir():
            if digit_folder.is_dir() and digit_folder.name.isdigit():
                d = int(digit_folder.name)
                for img_path in digit_folder.glob("*.*"):
                    if img_path.suffix.lower() in (".png", ".jpg", ".jpeg"):
                        val_samples.append((img_path, d))

    # 2. If real crops are not yet present, generate synthetic rolling-drum dataset
    if len(train_samples) == 0:
        print("No real crops found in data/digits/. Generating realistic synthetic rolling-drum samples...")
        synth_gen = SyntheticDigitGenerator(slot_height=IMG_HEIGHT, slot_width=IMG_WIDTH)
        for digit in range(10):
            for _ in range(80):  # 80 train crops per digit = 800 samples
                is_red = (random.random() < 0.25)  # 25% red drum (4th slot)
                offset_y = random.uniform(-0.35, 0.35) if random.random() < 0.3 else 0.0
                img = synth_gen.render_digit_slot(digit, is_red_drum=is_red, offset_y=offset_y)
                train_samples.append((img, digit))
            for _ in range(20):  # 20 val crops per digit = 200 samples
                is_red = (random.random() < 0.25)
                offset_y = random.uniform(-0.25, 0.25) if random.random() < 0.2 else 0.0
                img = synth_gen.render_digit_slot(digit, is_red_drum=is_red, offset_y=offset_y)
                val_samples.append((img, digit))

    random.shuffle(train_samples)
    random.shuffle(val_samples)
    print(f"Loaded {len(train_samples)} training samples and {len(val_samples)} validation samples.")
    return DialDigitDataset(train_samples, train_transform), DialDigitDataset(val_samples, val_transform)

train_dataset, val_dataset = prepare_datasets()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


--- 
### 3. Model Definition (`DigitANN`)
Pure PyTorch Multi-Layer Perceptron with Batch Normalization, ReLU activations, and Dropout for regularization.

In [ ]:
class DigitANN(nn.Module):
    def __init__(self, input_dim=2048, hidden_layers=[256, 128, 64], num_classes=10, dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_layers:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout)
            ])
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # Flatten (Batch, 1, 64, 32) -> (Batch, 2048)
        x = x.view(x.size(0), -1)
        return self.net(x)

model = DigitANN(
    input_dim=INPUT_DIM,
    hidden_layers=HIDDEN_LAYERS,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"DigitANN Initialized | Trainable Parameters: {total_params:,}")

--- 
### 4. Interactive Training Loop
Runs training with per-epoch loss and accuracy metrics, saving the best checkpoint automatically.

In [ ]:
best_val_acc = 0.0
best_state_dict = None
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # --- Training Phase ---
    model.train()
    train_loss, train_corr, train_tot = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=-1)
        train_corr += (preds == labels).sum().item()
        train_tot += len(labels)

    epoch_train_loss = train_loss / train_tot
    epoch_train_acc = train_corr / train_tot

    # --- Validation Phase ---
    model.eval()
    val_loss, val_corr, val_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)

            val_loss += loss.item() * len(labels)
            preds = logits.argmax(dim=-1)
            val_corr += (preds == labels).sum().item()
            val_tot += len(labels)

    epoch_val_loss = val_loss / val_tot
    epoch_val_acc = val_corr / val_tot
    scheduler.step()

    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)

    is_best = epoch_val_acc > best_val_acc
    if is_best:
        best_val_acc = epoch_val_acc
        best_state_dict = model.state_dict().copy()

    # Print progress (Every 5 epochs, or when best is updated)
    if epoch == 1 or epoch % 5 == 0 or is_best:
        tag = " ★ (Best)" if is_best else ""
        print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.3f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.3f}{tag}")

print(f"\nTraining completed! Best Validation Accuracy: {best_val_acc:.1%}")

--- 
### 5. Save Model Checkpoint (`models/best_digit_ann.pt`)
Saves the trained weights along with architecture metadata directly to `models/best_digit_ann.pt`, matching the checkpoint structure expected by `src/infer.py`.

In [ ]:
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)
save_path = models_dir / "best_digit_ann.pt"

# Restore best weights before saving
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

torch.save({
    "model_state_dict": model.state_dict(),
    "config": {
        "data": {"input_size": [IMG_HEIGHT, IMG_WIDTH], "channels": CHANNELS},
        "model": {"hidden_layers": HIDDEN_LAYERS, "activation": "relu", "batchnorm": True, "dropout": DROPOUT}
    },
    "val_acc": best_val_acc,
    "param_count": total_params,
    "classes": list(range(10))
}, save_path)

print(f"✓ Checkpoint successfully saved to:\n  {save_path.resolve()}")

--- 
### 6. Visualizations & Sample Predictions Gallery
Plots training vs validation convergence curves and displays sample predictions with confidence scores.

In [ ]:
# 1. Plot Training & Validation Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss", color="#1f77b4")
plt.plot(history["val_loss"], label="Val Loss", color="#ff7f0e")
plt.title("Cross-Entropy Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_acc"], label="Train Acc", color="#1f77b4")
plt.plot(history["val_acc"], label="Val Acc", color="#2ca02c")
plt.title("Classification Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 2. Sample Predictions Gallery
model.eval()
plt.figure(figsize=(14, 4))
sample_indices = random.sample(range(len(val_dataset)), min(10, len(val_dataset)))

with torch.no_grad():
    for i, idx in enumerate(sample_indices):
        img_tensor, true_label = val_dataset[idx]
        pred_logits = model(img_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(pred_logits, dim=-1)
        pred_label = probs.argmax().item()
        confidence = probs[0, pred_label].item()

        # Denormalize image for display
        disp_img = (img_tensor.squeeze().cpu().numpy() * 0.5 + 0.5).clip(0, 1)

        plt.subplot(1, 10, i + 1)
        plt.imshow(disp_img, cmap="gray")
        is_correct = (pred_label == true_label)
        color = "green" if is_correct else "red"
        plt.title(f"P:{pred_label} (T:{true_label})\n{confidence:.0%}", color=color, fontsize=10)
        plt.axis("off")

plt.suptitle("Sample Predictions on Held-Out Validation Slots (P=Pred, T=True)", fontsize=12, y=1.05)
plt.tight_layout()
plt.show()